In [ ]:
import os
import sys
import gzip
import pickle
import warnings
import logging

import numpy as np

from tqdm import TqdmSynchronisationWarning

warnings.simplefilter("ignore", TqdmSynchronisationWarning)

# multi-threaded: OMP/MKL thread count controlled by SLURM --cpus-per-task

In [ ]:
# ====== user paths ======
PATH_ROOT = "/Users/sijianfan/projects/BiSSGL"
DRIMC_PATH = os.path.join(PATH_ROOT, "scripts/methods/DRIMC")

# --- dataset selection (override with env var DATASET) ---
DATASET = os.environ.get("DATASET", "tuberculosis")
PATH_DATA = os.path.join(PATH_ROOT, "datasets/realAnalysis", DATASET)
PATH_OUTPUT = os.path.join(PATH_ROOT, "outputs/results/realAnalysis", DATASET, "noisy")
PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")

for p in (PATH_OUTPUT, PATH_ARCHIVE):
    os.makedirs(p, exist_ok=True)

filenames = {"output": "results_sgimc"}

In [ ]:
# ====== Python imports ======
sys.path.append(PATH_ROOT)

from sgimc.utils import mc_split, get_submatrix, load, save, sparsify_with_mask
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import ParameterGrid, ShuffleSplit
from scipy.sparse import coo_matrix

from sgimc import SparseGroupIMCClassifier

# ====== R setup via rpy2 (for CV splitting only) ======
import rpy2.robjects as robjects

r = robjects.r

r(f'setwd("{DRIMC_PATH}")')

# source only the CV helper
r('source("doCrossValidationByPairwise.R")')
doCrossValidationByPairwise_R = r["doCrossValidationByPairwise"]

In [ ]:
# ====== scoring helper (matches simulation scripts) ======
def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]

    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN

    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])

    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])

    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)

    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]


# ====== helper: numpy matrix → R matrix ======
def to_r_matrix(arr):
    return r.matrix(
        robjects.FloatVector(arr.flatten()),
        byrow=True,
        nrow=arr.shape[0],
        ncol=arr.shape[1],
    )


# ====== helper: extract fold data from R savedFolds ======
def extract_fold(savedFolds, trial, fold):
    """
    Extract training matrix, test positions, and test labels from R CV output.

    Returns
    -------
    Y_train  : ndarray (n, m)  dense 0/1 training matrix (test entries zeroed)
    test_row : ndarray (T,)    0-based row indices of test entries
    test_col : ndarray (T,)    0-based col indices of test entries
    test_label : ndarray (T,)  0/1 test labels
    known_drug_idx : ndarray   1-based indices of drugs appearing in training
    known_target_idx : ndarray 1-based indices of targets appearing in training
    """
    fold_data = savedFolds.rx2(trial + 1).rx2(fold + 1)  # R is 1-indexed
    Y_train = np.array(fold_data.rx2(7))
    test_label = np.array(fold_data.rx2(1)).flatten()
    test_row = np.array(fold_data.rx2(3)).flatten().astype(int) - 1  # → 0-based
    test_col = np.array(fold_data.rx2(4)).flatten().astype(int) - 1
    known_drug_idx = np.array(fold_data.rx2(5)).flatten().astype(int)
    known_target_idx = np.array(fold_data.rx2(6)).flatten().astype(int)
    return Y_train, test_row, test_col, test_label, known_drug_idx, known_target_idx


# ====== helper: build balanced pos/neg entry set for inner CV ======
def build_cv_entries(Y_train, rng, neg_ratio=1.0):
    """
    Select all positive entries and sample neg_ratio × #pos negative entries.

    Returns
    -------
    cv_rows   : ndarray   row indices of selected entries
    cv_cols   : ndarray   col indices of selected entries
    cv_labels : ndarray   0/1 labels
    """
    pos_rows, pos_cols = np.where(Y_train == 1)
    neg_rows, neg_cols = np.where(Y_train == 0)
    n_pos = len(pos_rows)
    n_neg = min(int(n_pos * neg_ratio), len(neg_rows))
    neg_sample = rng.choice(len(neg_rows), size=n_neg, replace=False)
    cv_rows = np.concatenate([pos_rows, neg_rows[neg_sample]])
    cv_cols = np.concatenate([pos_cols, neg_cols[neg_sample]])
    cv_labels = np.concatenate([np.ones(n_pos), np.zeros(n_neg)])
    return cv_rows, cv_cols, cv_labels

In [ ]:
# load required R packages
for pkg in ["matrixcalc", "data.table", "Rcpp", "ROCR", "Bolstad2", "MESS"]:
    r(f"library({pkg})")

R callback write-console: data.table 1.17.8 using 6 threads (see ?getDTthreads).    
R callback write-console: Latest news: r-datatable.com
  


In [ ]:
# ====== load dataset ======
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logging.info("Loading dataset: %s", DATASET)

filename_input = os.path.join(PATH_DATA, "cv_data", "staged_dataset.gz")
X, Y_feat, R_full = load(filename_input)

# adjacency matrix for R's CV function (dense 0/1)
Y_adj = (R_full > 0).astype(float)

# ====== create outer CV folds via R ======
kfold = 10
numSplit = 5
seeds_list = [7771, 8367, 22, 1812, 4659]
seeds_R = robjects.IntVector(seeds_list)

logging.info(
    "Creating %d trials × %d-fold pairwise CV splits via R ...", numSplit, kfold
)
savedFolds = doCrossValidationByPairwise_R(
    to_r_matrix(Y_adj), kfold=kfold, numSplit=numSplit, seeds=seeds_R
)

2026-04-16 20:48:52,865 INFO: Loading dataset: tuberculosis
2026-04-16 20:48:52,875 INFO: Creating 5 trials × 10-fold pairwise CV splits via R ...


In [ ]:
# ====== parameter grid ======
grid_model = ParameterGrid(
    {
        "C_lasso": [1.0, 1e-1, 1e-2],
        "C_group": [1.0, 1e-1, 1e-2],
        "C_ridge": [1.0, 1e-1, 1e-2],
        "rank": [25],
    }
)

# inner CV settings
N_INNER_SPLITS = 3
INNER_VAL_SIZE = 0.20

# ====== flatten combinations into a list ======
combos = []
for trial in range(numSplit):
    for fold in range(kfold):
        for i_m, par_mdl in enumerate(grid_model):
            combos.append((trial, fold, i_m, par_mdl))
n_combos = len(combos)

In [ ]:
# ====== flatten combinations into a list ======
combos = []
for trial in range(numSplit):
    for fold in range(kfold):
        for i_m, par_mdl in enumerate(grid_model):
            combos.append((trial, fold, i_m, par_mdl))
n_combos = len(combos)

In [9]:
n_combos

1350

In [ ]:
BASE_SEED = int(os.environ.get("BASE_SEED", str(0x0BADCAFE)))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [local] %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logging.info("LOCAL mode — running all %d combos", n_combos)

task_results = []

# optional tqdm progress bar
try:
    from tqdm.auto import tqdm

    _iter = tqdm(range(n_combos), desc="combos", total=n_combos)
except ImportError:
    _iter = range(n_combos)

2026-04-16 20:50:13,563 INFO: LOCAL mode — running all 1350 combos
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
combos:   0%|          | 0/1350 [00:00<?, ?it/s]

In [15]:
_iter = tqdm(range(1), desc="combos", total=1)

combos:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ====== run all combos ======
for combo_idx in _iter:
    trial, fold, i_m, par_mdl = combos[combo_idx]
    logging.info(
        "Running combo_idx=%d (trial=%d, fold=%d, rank=%d, "
        "C_lasso=%s, C_group=%s, C_ridge=%s)",
        combo_idx,
        trial,
        fold,
        par_mdl["rank"],
        par_mdl["C_lasso"],
        par_mdl["C_group"],
        par_mdl["C_ridge"],
    )

    # ------------------------------------------------------------------
    # Split RNG: depends only on (trial, fold) so that the same balanced
    # neg sample / inner splits are used across all HP combos.
    # ------------------------------------------------------------------
    split_seed = BASE_SEED + trial * 100 + fold
    rng_split = np.random.RandomState(split_seed)

    model_seed = BASE_SEED + combo_idx

    try:
        # ====== extract fold data ======
        Y_train_dense, test_row, test_col, test_label, _, _ = extract_fold(
            savedFolds, trial, fold
        )

        # extract model params
        C_lasso = par_mdl["C_lasso"]
        C_group = par_mdl["C_group"]
        C_ridge = par_mdl["C_ridge"]
        rank = par_mdl["rank"]

        # ====== fit on full training → test scores ======
        # SGIMC uses {-1, +1} encoding: 1→+1, 0→-1
        Y_train_signed = Y_train_dense.copy()
        Y_train_signed[Y_train_signed == 0] = -1
        R_train_full = coo_matrix(Y_train_signed)

        model = SparseGroupIMCClassifier(
            rank,
            n_threads=-1,
            random_state=model_seed,
            C_lasso=C_lasso,
            C_group=C_group,
            C_ridge=C_ridge,
        )
        model.fit(X, Y_feat, R_train_full)

        prob_full = model.predict_proba(X, Y_feat)
        if hasattr(prob_full, "toarray"):
            prob_full = prob_full.toarray()
        prob_test = prob_full[test_row, test_col]
        scores_test = get_metrics(
            np.asarray(test_label, dtype=float),
            np.asarray(prob_test, dtype=float),
        )
        d1_test = int(sum(abs(model.coef_W_).max(axis=1) > 0))
        d2_test = int(sum(abs(model.coef_H_).max(axis=1) > 0))
        # indices of selected features (non-zero rows of W and H)
        sel_idx_W = np.where(abs(model.coef_W_).max(axis=1) > 0)[0].tolist()
        sel_idx_H = np.where(abs(model.coef_H_).max(axis=1) > 0)[0].tolist()

        # ====== inner CV — balanced ShuffleSplit ======
        cv_rows, cv_cols, cv_labels = build_cv_entries(
            Y_train_dense, rng_split, neg_ratio=1.0
        )

        splt = ShuffleSplit(
            n_splits=N_INNER_SPLITS,
            test_size=INNER_VAL_SIZE,
            random_state=rng_split,
        )
        for cv, (ind_train, ind_valid) in enumerate(
            splt.split(np.arange(len(cv_labels)))
        ):
            # build inner training matrix: zero out validation positives
            Y_inner = Y_train_dense.copy()
            valid_pos_mask = cv_labels[ind_valid] == 1
            Y_inner[
                cv_rows[ind_valid[valid_pos_mask]],
                cv_cols[ind_valid[valid_pos_mask]],
            ] = 0

            # convert to SGIMC encoding
            Y_inner_signed = Y_inner.copy()
            Y_inner_signed[Y_inner_signed == 0] = -1
            R_inner = coo_matrix(Y_inner_signed)

            model_cv = SparseGroupIMCClassifier(
                rank,
                n_threads=-1,
                random_state=model_seed,
                C_lasso=C_lasso,
                C_group=C_group,
                C_ridge=C_ridge,
            )
            model_cv.fit(X, Y_feat, R_inner)

            prob_full_cv = model_cv.predict_proba(X, Y_feat)
            if hasattr(prob_full_cv, "toarray"):
                prob_full_cv = prob_full_cv.toarray()
            prob_valid = prob_full_cv[cv_rows[ind_valid], cv_cols[ind_valid]]
            scores_valid = get_metrics(
                np.asarray(cv_labels[ind_valid], dtype=float),
                np.asarray(prob_valid, dtype=float),
            )
            d1_valid = int(sum(abs(model_cv.coef_W_).max(axis=1) > 0))
            d2_valid = int(sum(abs(model_cv.coef_H_).max(axis=1) > 0))

            task_results.append(
                {
                    # --- fold identity ---
                    "trial": trial,
                    "fold": fold,
                    # --- hyperparameters ---
                    "C_lasso": C_lasso,
                    "C_group": C_group,
                    "C_ridge": C_ridge,
                    "rank": rank,
                    # --- inner CV fold index ---
                    "cv": int(cv),
                    # --- validation scores (for HP selection) ---
                    "val_score": scores_valid,
                    "val_d1": d1_valid,
                    "val_d2": d2_valid,
                    # --- test scores (reported after HP selection) ---
                    "test_score": scores_test,
                    "test_d1": d1_test,
                    "test_d2": d2_test,
                    # --- selected feature indices (test model) ---
                    "test_sel_W": sel_idx_W,
                    "test_sel_H": sel_idx_H,
                }
            )

    except Exception as e:
        logging.exception("Error at combo_idx=%d: %s", combo_idx, str(e))

2026-04-16 20:51:57,699 INFO: Running combo_idx=0 (trial=0, fold=0, rank=25, C_lasso=1.0, C_group=1.0, C_ridge=1.0)
combos: 100%|██████████| 1/1 [05:28<00:00, 328.95s/it]


In [19]:
len(task_results)

3